# Self-attention step by step

*Corrected version (Chapter 19). The original is kept as `attention_mechanisms_original.ipynb`.*  Six tokens with three-dimensional embeddings ("Your journey starts with one step").

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

inputs = torch.tensor(
  [[0.43, 0.15, 0.89],   # Your     (x^1)
   [0.55, 0.87, 0.66],   # journey  (x^2)
   [0.57, 0.85, 0.64],   # starts   (x^3)
   [0.22, 0.58, 0.33],   # with     (x^4)
   [0.77, 0.25, 0.10],   # one      (x^5)
   [0.05, 0.80, 0.55]])  # step     (x^6)

## 1. Attention scores for one query (the 2nd token)

The score is the dot product of the query with every token.

In [ ]:
query = inputs[1]
attn_scores_2 = inputs @ query            # same as the loop with torch.dot
print(attn_scores_2)

## 2. From scores to weights

Normalising by the sum works but is not used (scores can be negative). The softmax is positive and sums to one. The naive softmax `exp(x)/sum(exp(x))` overflows for large scores, so the library version subtracts the maximum first.

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

w_naive = softmax_naive(attn_scores_2)
w = torch.softmax(attn_scores_2, dim=0)
print('naive  :', w_naive)
print('library:', w, ' sum =', w.sum().item())

big = torch.tensor([1000., 1001., 1002.])
print('naive on large scores  :', softmax_naive(big))          # nan: exp overflows
print('library on large scores:', torch.softmax(big, dim=0))

## 3. The context vector of one token

In [ ]:
context_vec_2 = (w.unsqueeze(1) * inputs).sum(dim=0)         # = w @ inputs
print(context_vec_2)

## 4. All tokens at once

The scores of all queries against all keys are one matrix product. Each row of the weight matrix sums to 1.

In [ ]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)
print('row sums:', attn_weights.sum(dim=-1))       # computed, not typed in by hand
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)
print('row 2 equals the single-query result:', torch.allclose(all_context_vecs[1], context_vec_2))

## 5. Trainable weights: queries, keys, values

Three matrices project each token. The scores are divided by $\sqrt{d_k}$ before the softmax (the variance of a dot product of $d_k$ unit-variance numbers is $d_k$).

In [ ]:
d_in, d_out = 3, 2
torch.manual_seed(123)
W_query = torch.rand(d_in, d_out)
W_key   = torch.rand(d_in, d_out)
W_value = torch.rand(d_in, d_out)

queries, keys, values = inputs @ W_query, inputs @ W_key, inputs @ W_value
print('keys.shape:', keys.shape, ' values.shape:', values.shape)

d_k = keys.shape[1]
scores = queries @ keys.T
weights = torch.softmax(scores / d_k**0.5, dim=-1)
context = weights @ values
print(weights)
print(context)

## 6. A reusable module

`SelfAttention_v1` uses raw parameters, `SelfAttention_v2` uses `nn.Linear`. They give **different** numbers for the same input only because `nn.Linear` initialises the weights differently; with the same weights the outputs are identical (checked below).

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        q, k, v = x @ self.W_query, x @ self.W_key, x @ self.W_value
        w = torch.softmax(q @ k.T / k.shape[-1]**0.5, dim=-1)
        return w @ v

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        q, k, v = self.W_query(x), self.W_key(x), self.W_value(x)
        w = torch.softmax(q @ k.T / k.shape[-1]**0.5, dim=-1)
        return w @ v

torch.manual_seed(123); sa_v1 = SelfAttention_v1(d_in, d_out)
torch.manual_seed(789); sa_v2 = SelfAttention_v2(d_in, d_out)
print('v1:', sa_v1(inputs)[0])
print('v2:', sa_v2(inputs)[0])

# copy the v1 weights into v2 (nn.Linear stores W transposed): the outputs agree
with torch.no_grad():
    sa_v2.W_query.weight.copy_(sa_v1.W_query.T)
    sa_v2.W_key.weight.copy_(sa_v1.W_key.T)
    sa_v2.W_value.weight.copy_(sa_v1.W_value.T)
print('same weights -> same output:', torch.allclose(sa_v1(inputs), sa_v2(inputs), atol=1e-6))

## 7. Causal (masked) attention

A language model must not see future tokens. Two ways to get the same result:
1. multiply the weights by a lower-triangular mask and renormalise;
2. fill the scores above the diagonal with $-\infty$ **before** the softmax (better: one step, no renormalisation).

In [ ]:
queries, keys = sa_v2.W_query(inputs), sa_v2.W_key(inputs)
scores = queries @ keys.T
T = scores.shape[0]
weights = torch.softmax(scores / keys.shape[-1]**0.5, dim=-1)

# way 1
mask_simple = torch.tril(torch.ones(T, T))
masked_simple = weights * mask_simple
way1 = masked_simple / masked_simple.sum(dim=-1, keepdim=True)

# way 2
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
way2 = torch.softmax(scores.masked_fill(mask, -torch.inf) / keys.shape[-1]**0.5, dim=-1)

print(way2)
print('both ways agree:', torch.allclose(way1, way2, atol=1e-6))

## 8. Check against PyTorch's built-in attention

In [ ]:
q, k, v = sa_v2.W_query(inputs), sa_v2.W_key(inputs), sa_v2.W_value(inputs)
ref = F.scaled_dot_product_attention(q[None], k[None], v[None], is_causal=True)[0]
mine = way2 @ v
print('equals F.scaled_dot_product_attention(is_causal=True):', torch.allclose(mine, ref, atol=1e-6))